# Lain Brain × Apertus：首轮 zip 训练入口

先搭建，再开 GPU。这个 Notebook 内嵌校验过的脚本，不用解压或 clone。
第 1–3 步不下载模型、不调用老师、不训练。收集和训练步骤默认关闭。
已用随机微型 Apertus 和 Qwen2 检查前向与真实梯度；真实模型/GPU 尚未运行。

首轮：学生初稿 → Apertus 一个提示 → 学生修订 → 程序核验 → zip → 短训练。
老师意见不是证明；自然语言解释尚无自动真值证书。
这是接口实验，尚未实现研究调度/暂停重访，也不证明效果或成本优势。


## 1. 准备脚本（CPU 即可；不联网）

In [ ]:
from pathlib import Path
import base64, hashlib, json, subprocess, sys, uuid, os

WORK = Path.cwd() / 'lain-zip-pilot'
TOOLS = WORK / 'tools'
TOOLS.mkdir(parents=True, exist_ok=True)
PAYLOAD = json.loads(r'''{"zip_pilot.py": {"sha256": "72031cd0db0f2189b379f306a8f0724b03be9dc5ba231dffff03e898de7b4945", "base64": "..."}, "zip_pilot_protocol.py": {"sha256": "f7c9df1574ebaff35494397b7e1726877ee019b8d0afdfc109efe44d1d18f819", "base64": "IiIiQm91bmRlZCBtYXRoZW1hdGljYWwgb2JqZWN0cyBhbmQgYSBjYWNoZWQgQXBlcnR1cyBjcml0aXF1ZSB0cmFuc3BvcnQuCgpPbmx5IHBvbHlub21pYWwgY2xhaW1zIGFyZSBjZXJ0aWZpZWQuIFJlYWRhYmxlIHByb3NlIGFuZCB0ZWFjaGVyIGFncmVlbWVudCBhcmUKbm90IHByb29mcy4gVGhpcyBwaWxvdCBkb2VzIG5vdCBqdWRnZSBvcGVuIHJlc2VhcmNoIG9yIGxlYXJuIHRhc2sgc2NoZWR1bGluZy4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhc3QKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gZnJhY3Rpb25zIGltcG9ydCBGcmFjdGlvbiBhcyBGCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHJlCmltcG9ydCB0aW1lCmZyb20gdXJsbGliLnBhcnNlIGltcG9ydCB1cmxzcGxpdApmcm9tIHVybGxpYi5yZXF1ZXN0IGltcG9ydCBSZXF1ZXN0LCBidWlsZF9vcGVuZXIsIEhUVFBSZWRpcmVjdEhhbmRsZXIKCmZyb20gcmVzZWFyY2hfcmV3YXJkX3Byb2JlIGltcG9ydCBhZGQsIG11bCwgcG9seQoKUFJPVE9DT0wgPSAibGFpbi16aXAtcGlsb3QtdjIiClNDSEVNQSA9ICd7ImRlZmluaXRpb24iOiJ5b3VyIHNob3J0IGV4cGxhbmF0aW9uIiwgImV4cHJlc3Npb24iOiJleHBhbmRlZCBwb2x5bm9taWFsIiwgInNjb3BlIjoiZXhhY3QiLCAicmFkaXVzIjoiMC4xIiwgInN0YXR1cyI6ImFzc2VydGVkIn0nCgoKZGVmIGNhbm9uaWNhbChkYXRhKToKICAgIHJldHVybiBqc29uLmR1bXBzKGRhdGEsIGVuc3VyZV9hc2NpaT1GYWxzZSwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSwgYWxsb3dfbmFuPUZhbHNlKQoKCmRlZiBmaW5nZXJwcmludChkYXRhKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihjYW5vbmljYWwoZGF0YSkuZW5jb2RlKCkpLmhleGRpZ2VzdCgpCgoKZGVmIHdyaXRlX2pzb24ocGF0aCwgZGF0YSk6CiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigieCIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIHN0cmVhbToKICAgICAgICBzdHJlYW0ud3JpdGUoanNvbi5kdW1wcyhkYXRhLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yLCBhbGxvd19uYW49RmFsc2UpICsgIlxuIikKCgpkZWYgc3RyaWN0X2pzb24odGV4dCk6CiAgICAiIiJObyBleHRyYWN0aW5nIGEgY29udmVuaWVudCBKU09OIGZyYWdtZW50LCBOYU5zIG9yIGR1cGxpY2F0ZSBrZXlzLiIiIgogICAgaWYgbm90IGlzaW5zdGFuY2UodGV4dCwgc3RyKSBvciBsZW4odGV4dCkgPiAxNjAwMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJSZXNwb25zZSBtdXN0IGJlIGEgYm91bmRlZCBKU09OIHN0cmluZyIpCiAgICBkZWYgcGFpcnMoaXRlbXMpOgogICAgICAgIHJlc3VsdCA9IHt9CiAgICAgICAgZm9yIGtleSwgdmFsdWUgaW4gaXRlbXM6CiAgICAgICAgICAgIGlmIGtleSBpbiByZXN1bHQ6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEdXBsaWNhdGUgSlNPTiBrZXkiKQogICAgICAgICAgICByZXN1bHRba2V5XSA9IHZhbHVlCiAgICAgICAgcmV0dXJuIHJlc3VsdAogICAgZGVmIGJhZF9jb25zdGFudCh2YWx1ZSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTm9uLWZpbml0ZSBKU09OIHZhbHVlIikKICAgIHZhbHVlID0ganNvbi5sb2Fkcyh0ZXh0LCBvYmplY3RfcGFpcnNfaG9vaz1wYWlycywgcGFyc2VfY29uc3RhbnQ9YmFkX2NvbnN0YW50KQogICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIGRpY3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4cGVjdGVkIGEgSlNPTiBvYmplY3QiKQogICAgcmV0dXJuIHZhbHVlCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVGFzazoKICAgIGtleTogc3RyCiAgICBzcGxpdDogc3RyCiAgICBmYW1pbHk6IHN0cgogICAgc2NhbGU6IGludAoKICAgIGRlZiBwdWJsaWMoc2VsZik6CiAgICAgICAgYSA9IHNlbGYuc2NhbGUKICAgICAgICBpZiBzZWxmLmZhbWlseSA9PSAic3F1YXJlIjoKICAgICAgICAgICAgcHJvYmxlbSA9IGYiRm9yIGYoeCk9e2F9KngqKjIsIG1vZGVsIGYoeCt1KS1mKHgpLiBHaXZlIGFuIGV4cGFuZGVkIGV4YWN0IHBvbHlub21pYWwsIG9yIGEgbG9jYWwgYXBwcm94aW1hdGlvbiB2YWxpZCBmb3IgZXZlcnkgcmVhbCB4IGFuZCB8dXw8PXJhZGl1cyB3aXRoIGFic29sdXRlIGVycm9yIDw9MS8xMC4iCiAgICAgICAgZWxpZiBzZWxmLmZhbWlseSA9PSAicHJvZHVjdCI6CiAgICAgICAgICAgIHByb2JsZW0gPSBmIkZvciBmKHgseSk9e2F9KngqeSwgbW9kZWwgZih4K3UseSt2KS1mKHgseSkgYXMgYW4gZXhwYW5kZWQgZXhhY3QgcG9seW5vbWlhbC4iCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJvYmxlbSA9IGYiRm9yIGYoeCx5KT17YX0qKHgqeSkqKjIsIG1vZGVsIGYoeCt1LHkrdiktZih4LHkpIGFzIGFuIGV4cGFuZGVkIGV4YWN0IHBvbHlub21pYWwuIgogICAgICAgIHJldHVybiBkaWN0KGtleT1zZWxmLmtleSwgcHJvYmxlbT1wcm9ibGVtKQoKICAgIGRlZiB0YXJnZXQoc2VsZik6CiAgICAgICAgYSA9IHBvbHkoc3RyKHNlbGYuc2NhbGUpKQogICAgICAgIGV4cHJlc3Npb24gPSB7InNxdWFyZSI6ICIoeCt1KSoqMi14KioyIiwgInByb2R1Y3QiOiAiKHgrdSkqKHkrdikteCp5IiwKICAgICAgICAgICAgICAgICAgICAgICJjb21wb3NpdGlvbiI6ICIoKHgrdSkqKHkrdikpKioyLSh4KnkpKioyIn1bc2VsZi5mYW1pbHldCiAgICAgICAgcmV0dXJuIG11bChhLCBwb2x5KGV4cHJlc3Npb24pKQoKCmRlZiB0YXNrcyhzcGxpdCk6CiAgICAiIiJQdWJsaWMgZGV2ZWxvcG1lbnQgYmVuY2htYXJrLiBGaW5hbCBzcGxpdCBpcyBuZXZlciBzZW50IHRvIHRoZSB0ZWFjaGVyLiIiIgogICAgZ3JvdXBzID0geyJ0cmFpbiI6IChyYW5nZSgxLCA0KSwgKCJzcXVhcmUiLCAicHJvZHVjdCIpKSwKICAgICAgICAgICAgICAiZGV2IjogKHJhbmdlKDQsIDYpLCAoInNxdWFyZSIsICJwcm9kdWN0IikpLAogICAgICAgICAgICAgICJmaW5hbCI6IChyYW5nZSg2LCA4KSwgKCJzcXVhcmUiLCAicHJvZHVjdCIsICJjb21wb3NpdGlvbiIpKX0KICAgIHZhbHVlcywgZmFtaWxpZXMgPSBncm91cHNbc3BsaXRdCiAgICByZXR1cm4gW1Rhc2soZiJ7c3BsaXR9LXtmYW1pbHl9LXthfSIsIHNwbGl0LCBmYW1pbHksIGEpIGZvciBmYW1pbHkgaW4gZmFtaWxpZXMgZm9yIGEgaW4gdmFsdWVzXQoKCmRlZiBwcm9tcHQodGFzaywgZmVlZGJhY2s9Tm9uZSwgbm90ZXM9Tm9uZSk6CiAgICBtZXNzYWdlcyA9IFt7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiAiTW9kZWwgYSBtYXRoZW1hdGljYWwgb2JqZWN0IGluIHlvdXIgb3duIHdvcmRzLiBPdXRwdXQgb25seSBhIEpTT04gb2JqZWN0IHVzaW5nIHRoaXMgc2NoZW1hOiAiICsgU0NIRU1BICsgIi4gUmVwbGFjZSB0aGUgZGVmaW5pdGlvbiBhbmQgZXhwcmVzc2lvbiBwbGFjZWhvbGRlcnMgd2l0aCB5b3VyIG1vZGVsLiBDaG9vc2Ugb25lIHNjb3BlIHZhbHVlOiBleGFjdCBvciBsb2NhbC4gQ2hvb3NlIG9uZSBzdGF0dXMgdmFsdWU6IGFzc2VydGVkIG9yIGNvbmplY3R1cmU7IG5ldmVyIGNvcHkgdGhlIGNvbWJpbmVkIGFsdGVybmF0aXZlcy4gQWxsIGZpdmUgdmFsdWVzIGFyZSBzdHJpbmdzLiBLZWVwIGRlZmluaXRpb24gdG8gb25lIHNob3J0IHNlbnRlbmNlIGFuZCBjbG9zZSB0aGUgSlNPTiB3aXRoIH0uIFZhcmlhYmxlcyBhcmUgeCx5LHUsdi4gVXNlICoqIGZvciBwb3dlcnMsIG5ldmVyIF4uIE5vIGZ1bmN0aW9uIGNhbGxzLiBFeGFjdCBmb3JtdWxhcyBtdXN0IGJlIGV4cGFuZGVkIHN1bXMgb2YgbW9ub21pYWxzOyByZXN0YXRpbmcgdGhlIG9yaWdpbmFsIGRpZmZlcmVuY2UgaXMgbm90IGEgbW9kZWwuIE1hcmsgdW5zdXBwb3J0ZWQgZ3Vlc3NlcyBhcyBjb25qZWN0dXJlLiBOYXR1cmFsLWxhbmd1YWdlIGV4cGxhbmF0aW9ucyBhcmUgcmV2aWV3ZWQgc2VwYXJhdGVseS4ifV0KICAgIGJvZHkgPSB0YXNrLnB1YmxpYygpWyJwcm9ibGVtIl0KICAgIGlmIG5vdGVzOgogICAgICAgIGJvZHkgKz0gIlxuVHJhaW5pbmcgbm90ZXMgKHRoZWlyIGZvcm11bGFzIGhhdmUgY2VydGlmaWNhdGVzOyBwcm9zZSBpcyBub3QgY2VydGlmaWVkKTpcbiIgKyAiXG4iLmpvaW4obm90ZXMpCiAgICBpZiBmZWVkYmFjazoKICAgICAgICBib2R5ICs9ICJcbkVhcmxpZXIgZHJhZnQgYW5kIGZlZWRiYWNrOlxuIiArIGNhbm9uaWNhbChmZWVkYmFjaykgKyAiXG5Qcm9kdWNlIHlvdXIgcmV2aXNlZCBtb2RlbC4iCiAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IGJvZHl9KQogICAgcmV0dXJuIG1lc3NhZ2VzCgoKZGVmIGV4cGFuZGVkKGV4cHJlc3Npb24pOgogICAgIiIiUmVxdWlyZSBleHBhbmRlZCBtb25vbWlhbHMgc28gY29weWluZyB0aGUgcXVlc3Rpb24gZWFybnMgbm8gY3JlZGl0LiIiIgogICAgdHJlZSA9IGFzdC5wYXJzZShleHByZXNzaW9uLCBtb2RlPSJldmFsIikKICAgIGRlZiB0ZXJtKG5vZGUpOgogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgYXN0Lk5hbWUpOgogICAgICAgICAgICByZXR1cm4gbm9kZS5pZCBpbiB7IngiLCAieSIsICJ1IiwgInYifQogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgYXN0LkNvbnN0YW50KToKICAgICAgICAgICAgcmV0dXJuIHR5cGUobm9kZS52YWx1ZSkgaXMgaW50CiAgICAgICAgaWYgaXNpbnN0YW5jZShub2RlLCBhc3QuVW5hcnlPcCkgYW5kIGlzaW5zdGFuY2Uobm9kZS5vcCwgKGFzdC5VU3ViLCBhc3QuVUFkZCkpOgogICAgICAgICAgICByZXR1cm4gdGVybShub2RlLm9wZXJhbmQpCiAgICAgICAgaWYgaXNpbnN0YW5jZShub2RlLCBhc3QuQmluT3ApIGFuZCBpc2luc3RhbmNlKG5vZGUub3AsIGFzdC5NdWx0KToKICAgICAgICAgICAgcmV0dXJuIHRlcm0obm9kZS5sZWZ0KSBhbmQgdGVybShub2RlLnJpZ2h0KQogICAgICAgIHJldHVybiAoaXNpbnN0YW5jZShub2RlLCBhc3QuQmluT3ApIGFuZCBpc2luc3RhbmNlKG5vZGUub3AsIGFzdC5Qb3cpCiAgICAgICAgICAgICAgICBhbmQgaXNpbnN0YW5jZShub2RlLmxlZnQsIGFzdC5OYW1lKSBhbmQgbm9kZS5sZWZ0LmlkIGluIHsieCIsICJ5IiwgInUiLCAidiJ9CiAgICAgICAgICAgICAgICBhbmQgaXNpbnN0YW5jZShub2RlLnJpZ2h0LCBhc3QuQ29uc3RhbnQpIGFuZCB0eXBlKG5vZGUucmlnaHQudmFsdWUpIGlzIGludCkKICAgIGRlZiB0ZXJtcyhub2RlKToKICAgICAgICBpZiBpc2luc3RhbmNlKG5vZGUsIGFzdC5CaW5PcCkgYW5kIGlzaW5zdGFuY2Uobm9kZS5vcCwgKGFzdC5BZGQsIGFzdC5TdWIpKToKICAgICAgICAgICAgcmV0dXJuIHRlcm1zKG5vZGUubGVmdCkgYW5kIHRlcm1zKG5vZGUucmlnaHQpCiAgICAgICAgcmV0dXJuIHRlcm0obm9kZSkKICAgIHJldHVybiB0ZXJtcyh0cmVlLmJvZHkpCgoKZGVmIHZlcmlmeSh0YXNrLCByYXcpOgogICAgcmVzdWx0ID0gZGljdChhY2NlcHRlZD1GYWxzZSwgc2NvcmU9MC4wLCBhc3NlcnRlZD1UcnVlLCBjZXJ0aWZpY2F0ZT1Ob25lLAogICAgICAgICAgICAgICAgICB0ZWFjaGVyQWdyZWVtZW50VXNlZD1GYWxzZSwgcHJvc2VDZXJ0aWZpZWQ9RmFsc2UpCiAgICB0cnk6CiAgICAgICAgY2xhaW0gPSBzdHJpY3RfanNvbihyYXcpCiAgICAgICAgaWYgc2V0KGNsYWltKSAhPSB7ImRlZmluaXRpb24iLCAiZXhwcmVzc2lvbiIsICJzY29wZSIsICJyYWRpdXMiLCAic3RhdHVzIn06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNjaGVtYSBmaWVsZHMgZG8gbm90IG1hdGNoIikKICAgICAgICBpZiBub3QgYWxsKGlzaW5zdGFuY2Uodiwgc3RyKSBmb3IgdiBpbiBjbGFpbS52YWx1ZXMoKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkFsbCBjbGFpbSBmaWVsZHMgbXVzdCBiZSBzdHJpbmdzIikKICAgICAgICBpZiBub3QgMSA8PSBsZW4oY2xhaW1bImRlZmluaXRpb24iXSkgPD0gMTIwMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwbGFuYXRpb24gbGVuZ3RoIGlzIGludmFsaWQiKQogICAgICAgIGlmIGNsYWltWyJzdGF0dXMiXSBub3QgaW4geyJhc3NlcnRlZCIsICJjb25qZWN0dXJlIn06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuc3VwcG9ydGVkIHN0YXR1cyIpCiAgICAgICAgaWYgY2xhaW1bInNjb3BlIl0gbm90IGluIHsiZXhhY3QiLCAibG9jYWwifToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5zdXBwb3J0ZWQgc2NvcGUiKQogICAgICAgIHJlc3VsdFsiYXNzZXJ0ZWQiXSA9IGNsYWltWyJzdGF0dXMiXSA9PSAiYXNzZXJ0ZWQiCiAgICAgICAgcmVzdWx0WyJjbGFpbSJdID0gY2xhaW0KICAgICAgICBjYW5kaWRhdGUgPSBwb2x5KGNsYWltWyJleHByZXNzaW9uIl0pCiAgICAgICAgaWYgbm90IGV4cGFuZGVkKGNsYWltWyJleHByZXNzaW9uIl0pOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJGb3JtdWxhIG11c3QgYmUgZXhwYW5kZWQsIG5vdCBhIHJlc3RhdGVtZW50IG9mIHRoZSBxdWVzdGlvbiIpCiAgICAgICAgaWYgbm90IHJlc3VsdFsiYXNzZXJ0ZWQiXToKICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJIb25lc3QgY29uamVjdHVyZTogbm90IGFkbWl0dGVkIGFzIGtub3dsZWRnZSIKICAgICAgICAgICAgcmV0dXJuIHJlc3VsdAogICAgICAgIGlmIGNsYWltWyJzY29wZSJdID09ICJleGFjdCI6CiAgICAgICAgICAgIHZhbGlkID0gY2FuZGlkYXRlID09IHRhc2sudGFyZ2V0KCkKICAgICAgICAgICAgc2NvcmUgPSAxLjAgaWYgdmFsaWQgZWxzZSAwLjAKICAgICAgICAgICAgY2VydGlmaWNhdGUgPSBkaWN0KGtpbmQ9ImV4YWN0LXBvbHlub21pYWwtY29lZmZpY2llbnRzIiwgY2xhaW1IYXNoPWZpbmdlcnByaW50KGNhbmRpZGF0ZV9yZXByKGNhbmRpZGF0ZSkpKQogICAgICAgIGVsaWYgY2xhaW1bInNjb3BlIl0gPT0gImxvY2FsIiBhbmQgdGFzay5mYW1pbHkgPT0gInNxdWFyZSI6CiAgICAgICAgICAgIGlmIG5vdCByZS5mdWxsbWF0Y2gociJbKy1dP1xkezEsNn0oPzpcLlxkezEsNn18L1xkezEsNn0pPyIsIGNsYWltWyJyYWRpdXMiXSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJSYWRpdXMgbXVzdCBiZSBhIGJvdW5kZWQgZGVjaW1hbCBvciBmcmFjdGlvbiwgbm90IGFuIGV4cG9uZW50IikKICAgICAgICAgICAgcmFkaXVzID0gRihjbGFpbVsicmFkaXVzIl0pCiAgICAgICAgICAgIHJlc2lkdWFsID0gYWRkKHRhc2sudGFyZ2V0KCksIGNhbmRpZGF0ZSwgLTEpCiAgICAgICAgICAgIHZhbGlkID0gcmVzaWR1YWwgPT0gcG9seShmInt0YXNrLnNjYWxlfSp1KioyIikgYW5kIHJhZGl1cyA+IDAKICAgICAgICAgICAgdmFsaWQgPSB2YWxpZCBhbmQgdGFzay5zY2FsZSAqIHJhZGl1cyoqMiA8PSBGKDEsIDEwKQogICAgICAgICAgICBzY29yZSA9ICgwLjQgaWYgcmFkaXVzID49IEYoMSwgMTApIGVsc2UgMC4xIGlmIHJhZGl1cyA+PSBGKDEsIDEwMCkgZWxzZSAwLjApIGlmIHZhbGlkIGVsc2UgMC4wCiAgICAgICAgICAgIGNlcnRpZmljYXRlID0gZGljdChraW5kPSJleGFjdC1yZXNpZHVhbC1yYXRpb25hbC1ib3VuZCIsIHJlc2lkdWFsPWYie3Rhc2suc2NhbGV9KnUqKjIiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmFkaXVzPXN0cihyYWRpdXMpLCBhYnNvbHV0ZVRvbGVyYW5jZT0iMS8xMCIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5zdXBwb3J0ZWQgc2NvcGUgZm9yIHRoaXMgb2JqZWN0IikKICAgICAgICByZXN1bHQudXBkYXRlKGFjY2VwdGVkPWJvb2wodmFsaWQpLCBzY29yZT1zY29yZSwgY2VydGlmaWNhdGU9Y2VydGlmaWNhdGUgaWYgdmFsaWQgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgICAgICAgcmVhc29uPSJGb3JtdWxhIGFuZCBkZWNsYXJlZCBzY29wZSBjZXJ0aWZpZWQiIGlmIHZhbGlkIGVsc2UgIkZvcm11bGEgb3Igc2NvcGUgZGlzcHJvdmVkIikKICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgU3ludGF4RXJyb3IsIFR5cGVFcnJvciwgT3ZlcmZsb3dFcnJvciwgWmVyb0RpdmlzaW9uRXJyb3IpIGFzIGVycm9yOgogICAgICAgIHJlc3VsdFsicmVhc29uIl0gPSBzdHIoZXJyb3IpCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGNhbmRpZGF0ZV9yZXByKGNhbmRpZGF0ZSk6CiAgICByZXR1cm4gW1tsaXN0KGtleSksIHN0cih2YWx1ZSldIGZvciBrZXksIHZhbHVlIGluIHNvcnRlZChjYW5kaWRhdGUuaXRlbXMoKSldCgoKZGVmIHRyYW5zaXRpb24odGFzaywgcmF3LCBwcmV2aW91cywgdG9rZW5zKToKICAgIGNoZWNrZWQgPSB2ZXJpZnkodGFzaywgcmF3KQogICAgYWZ0ZXIgPSBtYXgocHJldmlvdXMsIGNoZWNrZWRbInNjb3JlIl0pCiAgICBjb3N0ID0gMC4wMiAqIHRva2VucyAvIDI1NgogICAgcGVuYWx0eSA9IDAuMSBpZiBjaGVja2VkWyJhc3NlcnRlZCJdIGFuZCBub3QgY2hlY2tlZFsiYWNjZXB0ZWQiXSBlbHNlIDAuMAogICAgcmV0dXJuIGRpY3QodmVyaWZpY2F0aW9uPWNoZWNrZWQsIGJlZm9yZT1wcmV2aW91cywgYWZ0ZXI9YWZ0ZXIsCiAgICAgICAgICAgICAgICBnYWluPWFmdGVyIC0gcHJldmlvdXMsIGNvc3Q9Y29zdCwgdW5zdXBwb3J0ZWRQZW5hbHR5PXBlbmFsdHksCiAgICAgICAgICAgICAgICByZXdhcmQ9YWZ0ZXIgLSBwcmV2aW91cyAtIGNvc3QgLSBwZW5hbHR5KQoKCmNsYXNzIE5vUmVkaXJlY3QoSFRUUFJlZGlyZWN0SGFuZGxlcik6CiAgICBkZWYgcmVkaXJlY3RfcmVxdWVzdChzZWxmLCByZXEsIGZwLCBjb2RlLCBtc2csIGhlYWRlcnMsIG5ld3VybCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUZWFjaGVyIHJlZGlyZWN0IHJlamVjdGVkOyBjb25maWd1cmUgdGhlIGZpbmFsIGVuZHBvaW50IikKCgpjbGFzcyBUZWFjaGVyOgogICAgIiIiT3BlbkFJLWNvbXBhdGlibGUgY3JpdGlxdWUgZW5kcG9pbnQuIEV4cGxpY2l0IGJvdW5kZWQgcmVxdWVzdHMgb25seS4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBiYXNlX3VybCwgbW9kZWwsIHJldmlzaW9uLCBjYWNoZSwgKiwga2V5X2Vudj0iTEFJTl9URUFDSEVSX0FQSV9LRVkiLCBtYXhfY2FsbHM9Nik6CiAgICAgICAgcGFydHMgPSB1cmxzcGxpdChiYXNlX3VybCkKICAgICAgICBpZiAocGFydHMuc2NoZW1lIG5vdCBpbiB7Imh0dHAiLCAiaHR0cHMifSBvciBub3QgcGFydHMuaG9zdG5hbWUgb3IgcGFydHMudXNlcm5hbWUgb3IgcGFydHMucGFzc3dvcmQKICAgICAgICAgICAgICAgIG9yIHBhcnRzLnF1ZXJ5IG9yIHBhcnRzLmZyYWdtZW50IG9yIChwYXJ0cy5zY2hlbWUgIT0gImh0dHBzIiBhbmQgcGFydHMuaG9zdG5hbWUgbm90IGluIHsibG9jYWxob3N0IiwgIjEyNy4wLjAuMSIsICI6OjEifSkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUZWFjaGVyIFVSTCBtdXN0IGJlIEhUVFBTIG9yIGEgbG9jYWwgSFRUUCBzZXJ2ZXIsIHdpdGggbm8gVVJMIGNyZWRlbnRpYWxzIikKICAgICAgICBpZiAiYXBlcnR1cyIgbm90IGluIG1vZGVsLmxvd2VyKCkgb3Igbm90IHJldmlzaW9uIG9yIGxlbihyZXZpc2lvbikgPiAxMjg6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkRlY2xhcmUgdGhlIEFwZXJ0dXMgc2VydmVkIG1vZGVsIGFuZCBkZXBsb3ltZW50IHJldmlzaW9uIikKICAgICAgICBpZiBub3QgMSA8PSBtYXhfY2FsbHMgPD0gMTI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRlYWNoZXIgY2FsbCBidWRnZXQgbXVzdCBiZSBiZXR3ZWVuIDEgYW5kIDEyIikKICAgICAgICBzZWxmLnVybCA9IGJhc2VfdXJsLnJzdHJpcCgiLyIpICsgIi9jaGF0L2NvbXBsZXRpb25zIgogICAgICAgIHNlbGYubW9kZWwsIHNlbGYucmV2aXNpb24gPSBtb2RlbCwgcmV2aXNpb24KICAgICAgICBzZWxmLmtleSA9IG9zLmVudmlyb24uZ2V0KGtleV9lbnYsICIiKQogICAgICAgIHNlbGYuY2FjaGUgPSBQYXRoKGNhY2hlKQogICAgICAgIHNlbGYuY2FjaGUubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHNlbGYuY2FsbHMsIHNlbGYubWF4X2NhbGxzID0gMCwgbWF4X2NhbGxzCgogICAgZGVmIHJldmlldyhzZWxmLCB0YXNrLCBkcmFmdCwgdmVyaWZpY2F0aW9uKToKICAgICAgICBpZiB0YXNrLnNwbGl0ICE9ICJ0cmFpbiI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRlYWNoZXIgY2Fubm90IHNlZSBkZXYvZmluYWwgdGFza3MiKQogICAgICAgIHJlcXVlc3QgPSBkaWN0KG1vZGVsPXNlbGYubW9kZWwsIHRlbXBlcmF0dXJlPTAsIG1heF90b2tlbnM9MjU2LCBtZXNzYWdlcz1bCiAgICAgICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6ICdDcml0aXF1ZSB0aGUgc3R1ZGVudCBtb2RlbC4gT3V0cHV0IG9ubHkgSlNPTiB3aXRoIGZvdXIgc3RyaW5nIGZpZWxkczogYXNzZXNzbWVudCwgaGludCwgY291bnRlcmV4YW1wbGUsIHNjb3BlX25vdGUuIEdpdmUgb25lIHNob3J0IGhpbnQsIG5vdCBhIHJlcGxhY2VtZW50IGV4cGFuZGVkIGZvcm11bGEuIEFjY2VwdCB2YWxpZCBsb2NhbCBwcm9ncmVzcyBldmVuIHdoZW4gaW5jb21wbGV0ZS4gUHJvZ3JhbSBjZXJ0aWZpY2F0ZXMgY2hlY2sgZm9ybXVsYXMgb25seTsgdGVhY2hlciBhcHByb3ZhbCBuZXZlciBjZXJ0aWZpZXMgdHJ1dGguJ30sCiAgICAgICAgICAgIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBjYW5vbmljYWwoZGljdChvYmplY3Q9dGFzay5wdWJsaWMoKSwgZHJhZnQ9ZHJhZnQsIHByb2dyYW1DaGVjaz12ZXJpZmljYXRpb24pKX1dKQogICAgICAgIGtleSA9IGZpbmdlcnByaW50KGRpY3QocHJvdG9jb2w9UFJPVE9DT0wsIHVybD1zZWxmLnVybCwgZGVjbGFyZWRSZXZpc2lvbj1zZWxmLnJldmlzaW9uLCByZXF1ZXN0PXJlcXVlc3QpKQogICAgICAgIHBhdGggPSBzZWxmLmNhY2hlIC8gKGtleSArICIuanNvbiIpCiAgICAgICAgaWYgcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgY2FjaGVkID0ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dCgpKQogICAgICAgICAgICBpZiBjYWNoZWQuZ2V0KCJyZXF1ZXN0SGFzaCIpICE9IGtleSBvciBjYWNoZWQuZ2V0KCJyZXF1ZXN0IikgIT0gcmVxdWVzdDoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVGVhY2hlciBjYWNoZSBtaXNtYXRjaCIpCiAgICAgICAgICAgIHJldHVybiBkaWN0KGNhY2hlZCwgY2FjaGVIaXQ9VHJ1ZSkKICAgICAgICBpZiBzZWxmLmNhbGxzID49IHNlbGYubWF4X2NhbGxzOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRlYWNoZXIgY2FsbCBidWRnZXQgZXhoYXVzdGVkIikKICAgICAgICBoZWFkZXJzID0geyJDb250ZW50LVR5cGUiOiAiYXBwbGljYXRpb24vanNvbiJ9CiAgICAgICAgaWYgc2VsZi5rZXk6CiAgICAgICAgICAgIGhlYWRlcnNbIkF1dGhvcml6YXRpb24iXSA9ICJCZWFyZXIgIiArIHNlbGYua2V5CiAgICAgICAgc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICBzZWxmLmNhbGxzICs9IDEKICAgICAgICByZXNwb25zZSA9IGJ1aWxkX29wZW5lcihOb1JlZGlyZWN0KCkpLm9wZW4oUmVxdWVzdChzZWxmLnVybCwgZGF0YT1jYW5vbmljYWwocmVxdWVzdCkuZW5jb2RlKCksIGhlYWRlcnM9aGVhZGVycyksIHRpbWVvdXQ9NDUpCiAgICAgICAgd2l0aCByZXNwb25zZToKICAgICAgICAgICAgZGF0YSA9IHJlc3BvbnNlLnJlYWQoMjYyMTQ1KQogICAgICAgIGlmIGxlbihkYXRhKSA+IDI2MjE0NDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUZWFjaGVyIHJlc3BvbnNlIGV4Y2VlZHMgc2l6ZSBib3VuZCIpCiAgICAgICAgcGF5bG9hZCA9IHN0cmljdF9qc29uKGRhdGEuZGVjb2RlKCkpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBjb250ZW50ID0gcGF5bG9hZFsiY2hvaWNlcyJdWzBdWyJtZXNzYWdlIl1bImNvbnRlbnQiXQogICAgICAgIGV4Y2VwdCAoS2V5RXJyb3IsIEluZGV4RXJyb3IsIFR5cGVFcnJvcikgYXMgZXJyb3I6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiVGVhY2hlciBkaWQgbm90IHJldHVybiBjaGF0IGNvbnRlbnQiKSBmcm9tIGVycm9yCiAgICAgICAgcmVjb3JkID0gZGljdChyZXF1ZXN0SGFzaD1rZXksIHJlcXVlc3Q9cmVxdWVzdCwgZGVjbGFyZWRSZXZpc2lvbj1zZWxmLnJldmlzaW9uLAogICAgICAgICAgICAgICAgICAgICAgcmV2aXNpb25JbmRlcGVuZGVudGx5VmVyaWZpZWQ9RmFsc2UsIHJldHVybmVkTW9kZWw9cGF5bG9hZC5nZXQoIm1vZGVsIiksCiAgICAgICAgICAgICAgICAgICAgICByYXc9Y29udGVudCwgdXNhZ2U9cGF5bG9hZC5nZXQoInVzYWdlIiksIHNlY29uZHM9dGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQsCiAgICAgICAgICAgICAgICAgICAgICBjYWNoZUhpdD1GYWxzZSwgaGludExldmVsPTEsIGNyaXRpcXVlPU5vbmUpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBjcml0aXF1ZSA9IHN0cmljdF9qc29uKGNvbnRlbnQpCiAgICAgICAgICAgIGlmIHNldChjcml0aXF1ZSkgIT0geyJhc3Nlc3NtZW50IiwgImhpbnQiLCAiY291bnRlcmV4YW1wbGUiLCAic2NvcGVfbm90ZSJ9IG9yIG5vdCBhbGwoaXNpbnN0YW5jZSh2LCBzdHIpIGZvciB2IGluIGNyaXRpcXVlLnZhbHVlcygpKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNyaXRpcXVlIHNjaGVtYSBtaXNtYXRjaCIpCiAgICAgICAgICAgIHJlY29yZFsiY3JpdGlxdWUiXSA9IGNyaXRpcXVlCiAgICAgICAgZXhjZXB0IChWYWx1ZUVycm9yLCBUeXBlRXJyb3IpIGFzIGVycm9yOgogICAgICAgICAgICByZWNvcmRbInBhcnNlRXJyb3IiXSA9IHN0cihlcnJvcikKICAgICAgICB3cml0ZV9qc29uKHBhdGgsIHJlY29yZCkKICAgICAgICByZXR1cm4gcmVjb3JkCg=="}, "zip_pilot_model.py": {"sha256": "2530e390412848344b8beb64f1f282fee326669627c201bc49f45d4c5c839c14", "base64": "IiIiQSBmcm96ZW4gQXBlcnR1cyBvciBRd2VuMiB3aXRoIGEgYm91bmRlZCB6aXAgcmVhZCBwYXRoIGFuZCBhZGFwdGVyLgoKVGhlIHBpbG90IGF0dGVuZHMgdG8gYSBzbWFsbCBmcm96ZW4gdGV4dCBhcmNoaXZlLiBJdCBoYXMgbm8gZGlzY3JldGUgY2FsbGluZwpjb250cm9sbGVyIG9yIGxlYXJuZWQgcmVzZWFyY2ggc2NoZWR1bGVyLiBPbmx5IGFkZGVkIG1vZHVsZXMgYXJlIG9wdGltaXplZC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKCmRlZiBzdGF0ZV9kaWdlc3QobW9kdWxlKToKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIGZvciBuYW1lLCB2YWx1ZSBpbiBzb3J0ZWQobW9kdWxlLnN0YXRlX2RpY3QoKS5pdGVtcygpKToKICAgICAgICBkaWdlc3QudXBkYXRlKG5hbWUuZW5jb2RlKCkpCiAgICAgICAgdmFsdWUgPSB2YWx1ZS5kZXRhY2goKS5jcHUoKS5jb250aWd1b3VzKCkKICAgICAgICBkaWdlc3QudXBkYXRlKHN0cih2YWx1ZS5kdHlwZSkuZW5jb2RlKCkpCiAgICAgICAgZGlnZXN0LnVwZGF0ZShzdHIodHVwbGUodmFsdWUuc2hhcGUpKS5lbmNvZGUoKSkKICAgICAgICBkaWdlc3QudXBkYXRlKHZhbHVlLnJlc2hhcGUoLTEpLnZpZXcodG9yY2gudWludDgpLm51bXB5KCkudG9ieXRlcygpKQogICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQoKCmNsYXNzIFppcFBpbG90KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYmFzZSwgdG9rZW5pemVyLCBub3RlcywgKiwgcmFuaz0zMiwgc2xvdHM9MiwgbW9kZT0iemlwIik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgYmFzZS5jb25maWcubW9kZWxfdHlwZSBub3QgaW4geyJhcGVydHVzIiwgInF3ZW4yIn0gb3Igbm90IDEgPD0gbGVuKG5vdGVzKSA8PSAxMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTmVlZCBhbiBBcGVydHVzL1F3ZW4yIHN0dWRlbnQgYW5kIDEuLjEyIGNlcnRpZmllZCB0cmFpbmluZyBub3RlcyIpCiAgICAgICAgaWYgbW9kZSBub3QgaW4geyJ6aXAiLCAidGV4dCIsICJub25lIn06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVua25vd24gbWVtb3J5IG1vZGUiKQogICAgICAgIHNlbGYuYmFzZSwgc2VsZi50b2tlbml6ZXIsIHNlbGYubm90ZXMsIHNlbGYubW9kZSA9IGJhc2UsIHRva2VuaXplciwgbGlzdChub3RlcyksIG1vZGUKICAgICAgICBzZWxmLmJhc2UuZXZhbCgpLnJlcXVpcmVzX2dyYWRfKEZhbHNlKQogICAgICAgIHNlbGYuYmFzZS5jb25maWcudXNlX2NhY2hlID0gRmFsc2UKICAgICAgICBzZWxmLmJhc2UuZ3JhZGllbnRfY2hlY2twb2ludGluZ19kaXNhYmxlKCkKICAgICAgICBkID0gYmFzZS5jb25maWcuaGlkZGVuX3NpemUKICAgICAgICBkZXZpY2UgPSBuZXh0KGJhc2UucGFyYW1ldGVycygpKS5kZXZpY2UKICAgICAgICBzZWxmLnJhbmssIHNlbGYuc2xvdHMgPSByYW5rLCBzbG90cwogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBwb29sZWQgPSBbXQogICAgICAgICAgICBmb3Igbm90ZSBpbiBub3RlczoKICAgICAgICAgICAgICAgIGlkcyA9IHRva2VuaXplcihub3RlLCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UsIHJldHVybl90ZW5zb3JzPSJwdCIpWyJpbnB1dF9pZHMiXS50byhkZXZpY2UpCiAgICAgICAgICAgICAgICBpZiBpZHMuc2hhcGVbMV0gPiAyNTY6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVHJhaW5pbmcgbm90ZSBleGNlZWRzIDI1NiB0b2tlbnM7IG5vIHNpbGVudCB0cnVuY2F0aW9uIikKICAgICAgICAgICAgICAgIHBvb2xlZC5hcHBlbmQoYmFzZS5nZXRfaW5wdXRfZW1iZWRkaW5ncygpKGlkcykuZmxvYXQoKS5tZWFuKDEpWzBdKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJub3RlX2VtYmVkZGluZ3MiLCB0b3JjaC5zdGFjayhwb29sZWQpKQogICAgICAgIHNlbGYuZW5jb2RlciA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKGQsIHJhbmspLCBubi5TaUxVKCksIG5uLkxpbmVhcihyYW5rLCBzbG90cyAqIGQpKQogICAgICAgIHNlbGYucSA9IG5uLkxpbmVhcihkLCByYW5rLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuayA9IG5uLkxpbmVhcihkLCByYW5rLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYudiA9IG5uLkxpbmVhcihkLCByYW5rLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYucmVhZF9vdXQgPSBubi5MaW5lYXIocmFuaywgZCwgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLmdlbmVyYXRpb25fYWRhcHRlciA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKGQsIHJhbmspLCBubi5TaUxVKCksIG5uLkxpbmVhcihyYW5rLCBkKSkKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5yZWFkX291dC53ZWlnaHQsIHN0ZD0wLjAwMSkKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5nZW5lcmF0aW9uX2FkYXB0ZXJbLTFdLndlaWdodCwgc3RkPTAuMDAxKQogICAgICAgIG5uLmluaXQuemVyb3NfKHNlbGYuZ2VuZXJhdGlvbl9hZGFwdGVyWy0xXS5iaWFzKQogICAgICAgIHNlbGYudG8oZGV2aWNlKQogICAgICAgIHNlbGYuYWJsYXRpb24gPSAibm9ybWFsIgogICAgICAgIHNlbGYucmVhZF9jYWxscyA9IDAKICAgICAgICBzZWxmLmxheWVyX2luZGV4ID0gbGVuKGJhc2UubW9kZWwubGF5ZXJzKSAvLyAyCiAgICAgICAgc2VsZi5faG9vayA9IGJhc2UubW9kZWwubGF5ZXJzW3NlbGYubGF5ZXJfaW5kZXhdLnJlZ2lzdGVyX2ZvcndhcmRfaG9vayhzZWxmLl9yZWFkKQoKICAgIGRlZiB0cmFpbihzZWxmLCBtb2RlPVRydWUpOgogICAgICAgIHN1cGVyKCkudHJhaW4obW9kZSkKICAgICAgICBzZWxmLmJhc2UuZXZhbCgpICAjIG5vIGRyb3BvdXQgZGlzY3JlcGFuY3kgYmV0d2VlbiBzYW1wbGluZyBhbmQgcmVzY29yaW5nCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgX3JlYWQoc2VsZiwgbW9kdWxlLCBhcmdzLCBvdXRwdXQpOgogICAgICAgIGhpZGRlbiA9IG91dHB1dFswXSBpZiBpc2luc3RhbmNlKG91dHB1dCwgdHVwbGUpIGVsc2Ugb3V0cHV0CiAgICAgICAgaCA9IGhpZGRlbi5mbG9hdCgpCiAgICAgICAgZGVsdGEgPSBzZWxmLmdlbmVyYXRpb25fYWRhcHRlcihoKQogICAgICAgIGlmIHNlbGYubW9kZSA9PSAiemlwIiBhbmQgc2VsZi5hYmxhdGlvbiAhPSAib2ZmIjoKICAgICAgICAgICAgbWVtb3J5ID0gc2VsZi5lbmNvZGVyKHNlbGYubm90ZV9lbWJlZGRpbmdzKS5yZXNoYXBlKC0xLCBoaWRkZW4uc2hhcGVbLTFdKQogICAgICAgICAgICBrZXlzLCB2YWx1ZXMgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLm5vcm1hbGl6ZShzZWxmLmsobWVtb3J5KSwgZGltPS0xKSwgc2VsZi52KG1lbW9yeSkKICAgICAgICAgICAgaWYgc2VsZi5hYmxhdGlvbiA9PSAic2h1ZmZsZSI6CiAgICAgICAgICAgICAgICAjIENvbnRlbnQgbWlzbWF0Y2g6IHByZXNlcnZlIGtleXMsIHBlcm11dGUgdmFsdWVzIGFjcm9zcyBvYmplY3RzLgogICAgICAgICAgICAgICAgdmFsdWVzID0gdmFsdWVzLnJlc2hhcGUobGVuKHNlbGYubm90ZXMpLCBzZWxmLnNsb3RzLCAtMSkucm9sbCgxLCAwKS5yZXNoYXBlKC0xLCBzZWxmLnJhbmspCiAgICAgICAgICAgIHF1ZXJpZXMgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLm5vcm1hbGl6ZShzZWxmLnEoaCksIGRpbT0tMSkKICAgICAgICAgICAgd2VpZ2h0cyA9IChxdWVyaWVzIEAga2V5cy5UICogc2VsZi5yYW5rKiowLjUpLnNvZnRtYXgoLTEpCiAgICAgICAgICAgIGRlbHRhID0gZGVsdGEgKyBzZWxmLnJlYWRfb3V0KHdlaWdodHMgQCB2YWx1ZXMpCiAgICAgICAgICAgIHNlbGYucmVhZF9jYWxscyArPSAxCiAgICAgICAgcmVzdWx0ID0gaGlkZGVuICsgZGVsdGEudG8oaGlkZGVuLmR0eXBlKQogICAgICAgIHJldHVybiAocmVzdWx0LCkgKyBvdXRwdXRbMTpdIGlmIGlzaW5zdGFuY2Uob3V0cHV0LCB0dXBsZSkgZWxzZSByZXN1bHQKCiAgICBkZWYgcGFyYW1ldGVyc190b190cmFpbihzZWxmKToKICAgICAgICByZXR1cm4gW3AgZm9yIHAgaW4gc2VsZi5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkXQoKICAgIGRlZiBhZGFwdGVyX3N0YXRlKHNlbGYpOgogICAgICAgIHJldHVybiB7azogdi5kZXRhY2goKS5jcHUoKS5jb250aWd1b3VzKCkgZm9yIGssIHYgaW4gc2VsZi5zdGF0ZV9kaWN0KCkuaXRlbXMoKSBpZiBub3Qgay5zdGFydHN3aXRoKCJiYXNlLiIpfQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGlkcywga2VlcD0wKToKICAgICAgICByZXR1cm4gc2VsZi5iYXNlKGlucHV0X2lkcz1pZHMsIHVzZV9jYWNoZT1GYWxzZSwgbG9naXRzX3RvX2tlZXA9a2VlcCkubG9naXRzCgogICAgZGVmIGNsb3NlKHNlbGYpOgogICAgICAgIHNlbGYuX2hvb2sucmVtb3ZlKCkKCgpkZWYgZW5jb2RlX3Byb21wdChtb2RlbCwgbWVzc2FnZXMsIG1heF9wcm9tcHQ9MjMwNCk6CiAgICB0ZXh0ID0gbW9kZWwudG9rZW5pemVyLmFwcGx5X2NoYXRfdGVtcGxhdGUobWVzc2FnZXMsIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSkKICAgIGlkcyA9IG1vZGVsLnRva2VuaXplcih0ZXh0LCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UsIHJldHVybl90ZW5zb3JzPSJwdCIsIHJldHVybl90b2tlbl90eXBlX2lkcz1GYWxzZSlbImlucHV0X2lkcyJdLnRvKG5leHQobW9kZWwucGFyYW1ldGVycygpKS5kZXZpY2UpCiAgICBpZiBpZHMuc2hhcGVbMV0gPiBtYXhfcHJvbXB0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJQcm9tcHQgZXhjZWVkcyBib3VuZGVkIGNvbnRleHQgKHtpZHMuc2hhcGVbMV19ID4ge21heF9wcm9tcHR9KTsgbm8gc2lsZW50IHRydW5jYXRpb24iKQogICAgcmV0dXJuIGlkcwoKCmRlZiBlb3NfaWRzKG1vZGVsKToKICAgIHZhbHVlID0gbW9kZWwuYmFzZS5nZW5lcmF0aW9uX2NvbmZpZy5lb3NfdG9rZW5faWQKICAgIHJldHVybiBzZXQodmFsdWUgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgbGlzdCkgZWxzZSBbdmFsdWVdKQoKCmRlZiBnZW5lcmF0ZShtb2RlbCwgbWVzc2FnZXMsICosIG1heF9uZXdfdG9rZW5zPTEyOCwgc2FtcGxlPUZhbHNlKToKICAgIGlkcyA9IGVuY29kZV9wcm9tcHQobW9kZWwsIG1lc3NhZ2VzKQogICAgcHJlZml4ID0gaWRzLnNoYXBlWzFdCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmb3IgXyBpbiByYW5nZShtYXhfbmV3X3Rva2Vucyk6CiAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKGlkcywga2VlcD0xKVs6LCAtMV0uZmxvYXQoKQogICAgICAgICAgICBjaG9zZW4gPSB0b3JjaC5tdWx0aW5vbWlhbChsb2dpdHMuc29mdG1heCgtMSksIDEpIGlmIHNhbXBsZSBlbHNlIGxvZ2l0cy5hcmdtYXgoLTEsIGtlZXBkaW09VHJ1ZSkKICAgICAgICAgICAgaWRzID0gdG9yY2guY2F0KChpZHMsIGNob3NlbiksIDEpCiAgICAgICAgICAgIGlmIGNob3Nlbi5pdGVtKCkgaW4gZW9zX2lkcyhtb2RlbCk6CiAgICAgICAgICAgICAgICBicmVhawogICAgZ2VuZXJhdGVkID0gaWRzWzAsIHByZWZpeDpdCiAgICByZXR1cm4gZGljdChwcm9tcHRJZHM9aWRzWzAsIDpwcmVmaXhdLnRvbGlzdCgpLCBnZW5lcmF0ZWRJZHM9Z2VuZXJhdGVkLnRvbGlzdCgpLAogICAgICAgICAgICAgICAgcmF3PW1vZGVsLnRva2VuaXplci5kZWNvZGUoZ2VuZXJhdGVkLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpLAogICAgICAgICAgICAgICAgZ2VuZXJhdGVkVG9rZW5zPWxlbihnZW5lcmF0ZWQpLCB0cnVuY2F0ZWQ9Z2VuZXJhdGVkWy0xXS5pdGVtKCkgbm90IGluIGVvc19pZHMobW9kZWwpKQoKCmRlZiB0b2tlbl9sb2dwcyhtb2RlbCwgcHJvbXB0X2lkcywgYW5zd2VyX2lkcyk6CiAgICBkZXZpY2UgPSBuZXh0KG1vZGVsLnBhcmFtZXRlcnMoKSkuZGV2aWNlCiAgICBqb2luZWQgPSB0b3JjaC50ZW5zb3IoW3Byb21wdF9pZHMgKyBhbnN3ZXJfaWRzXSwgZGV2aWNlPWRldmljZSkKICAgIGxvZ2l0cyA9IG1vZGVsKGpvaW5lZCwga2VlcD1sZW4oYW5zd2VyX2lkcykrMSlbOiwgOi0xXS5mbG9hdCgpCiAgICB0YXJnZXRzID0gdG9yY2gudGVuc29yKFthbnN3ZXJfaWRzXSwgZGV2aWNlPWRldmljZSkKICAgIHJldHVybiBsb2dpdHMubG9nX3NvZnRtYXgoLTEpLmdhdGhlcigtMSwgdGFyZ2V0cy51bnNxdWVlemUoLTEpKS5zcXVlZXplKC0xKQoKCmRlZiBzZnRfbG9zcyhtb2RlbCwgbWVzc2FnZXMsIGFuc3dlcik6CiAgICBwcm9tcHRfaWRzID0gZW5jb2RlX3Byb21wdChtb2RlbCwgbWVzc2FnZXMpWzBdLnRvbGlzdCgpCiAgICB0YXJnZXQgPSBtb2RlbC50b2tlbml6ZXIoYW5zd2VyLCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpWyJpbnB1dF9pZHMiXQogICAgaWYgbGVuKHRhcmdldCkgPiAyNTYgb3Igbm90IHRhcmdldDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTRlQgYW5zd2VyIGV4Y2VlZHMgMjU2IHRva2VucyIpCiAgICBlb3MgPSBtb2RlbC50b2tlbml6ZXIuZW9zX3Rva2VuX2lkCiAgICBpZiBlb3MgaXMgbm90IE5vbmU6CiAgICAgICAgdGFyZ2V0LmFwcGVuZChlb3MpCiAgICByZXR1cm4gLXRva2VuX2xvZ3BzKG1vZGVsLCBwcm9tcHRfaWRzLCB0YXJnZXQpLm1lYW4oKQo="}, "research_reward_probe.py": {"sha256": "4fa71099207dddb1971c2289092ec0a0543d19e0a4b7495c4a5ed823c945dec3", "base64": "IiIiSGFuZC13cml0dGVuLCBleGFjdCByZXdhcmQvc2NoZWR1bGluZyBkcnkgcnVuLiBObyBzdHVkZW50IG9yIG5ldXJhbCB0cmFpbmluZy4KClJ1biB3aXRoIFB5dGhvbidzIHN0YW5kYXJkIGxpYnJhcnkuIEFsbCBjbGFpbXMgYmVsb3cgYXJlIGZpeHR1cmVzLCBub3QgZGlzY292ZXJpZXMuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGFzdApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gZnJhY3Rpb25zIGltcG9ydCBGcmFjdGlvbiBhcyBGCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKVkFSUyA9ICgieCIsICJ5IiwgInUiLCAidiIpClpFUk8gPSAoMCwgMCwgMCwgMCkKCgpkZWYgY2xlYW4ocCk6CiAgICByZXR1cm4ge2s6IHYgZm9yIGssIHYgaW4gcC5pdGVtcygpIGlmIHZ9CgoKZGVmIGFkZChhLCBiLCBzaWduPTEpOgogICAgYyA9IGRpY3QoYSkKICAgIGZvciBrZXksIHZhbHVlIGluIGIuaXRlbXMoKToKICAgICAgICBjW2tleV0gPSBjLmdldChrZXksIEYoMCkpICsgc2lnbiAqIHZhbHVlCiAgICByZXR1cm4gY2xlYW4oYykKCgpkZWYgbXVsKGEsIGIpOgogICAgYyA9IHt9CiAgICBmb3Iga2EsIHZhIGluIGEuaXRlbXMoKToKICAgICAgICBmb3Iga2IsIHZiIGluIGIuaXRlbXMoKToKICAgICAgICAgICAgayA9IHR1cGxlKGkgKyBqIGZvciBpLCBqIGluIHppcChrYSwga2IpKQogICAgICAgICAgICBpZiBzdW0oaykgPiA4OgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUG9seW5vbWlhbCBkZWdyZWUgZXhjZWVkcyBkcnktcnVuIGJvdW5kIikKICAgICAgICAgICAgY1trXSA9IGMuZ2V0KGssIEYoMCkpICsgdmEgKiB2YgogICAgcmV0dXJuIGNsZWFuKGMpCgoKZGVmIHBvbHkodGV4dCk6CiAgICBpZiBub3QgaXNpbnN0YW5jZSh0ZXh0LCBzdHIpIG9yIGxlbih0ZXh0KSA+IDI1NjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHByZXNzaW9uIG11c3QgYmUgYSBzaG9ydCBzdHJpbmciKQogICAgdHJlZSA9IGFzdC5wYXJzZSh0ZXh0LCBtb2RlPSJldmFsIikKICAgIGlmIHN1bSgxIGZvciBfIGluIGFzdC53YWxrKHRyZWUpKSA+IDEwMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHByZXNzaW9uIGV4Y2VlZHMgZHJ5LXJ1biBib3VuZCIpCgogICAgZGVmIHBhcnNlKG5vZGUpOgogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgYXN0Lk5hbWUpIGFuZCBub2RlLmlkIGluIFZBUlM6CiAgICAgICAgICAgIGtleSA9IHR1cGxlKGludChuYW1lID09IG5vZGUuaWQpIGZvciBuYW1lIGluIFZBUlMpCiAgICAgICAgICAgIHJldHVybiB7a2V5OiBGKDEpfQogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgYXN0LkNvbnN0YW50KSBhbmQgdHlwZShub2RlLnZhbHVlKSBpcyBpbnQgYW5kIGFicyhub2RlLnZhbHVlKSA8PSAxMDA6CiAgICAgICAgICAgIHJldHVybiBjbGVhbih7WkVSTzogRihub2RlLnZhbHVlKX0pCiAgICAgICAgaWYgaXNpbnN0YW5jZShub2RlLCBhc3QuVW5hcnlPcCkgYW5kIGlzaW5zdGFuY2Uobm9kZS5vcCwgKGFzdC5VU3ViLCBhc3QuVUFkZCkpOgogICAgICAgICAgICBzaWduID0gLTEgaWYgaXNpbnN0YW5jZShub2RlLm9wLCBhc3QuVVN1YikgZWxzZSAxCiAgICAgICAgICAgIHJldHVybiB7azogc2lnbiAqIHYgZm9yIGssIHYgaW4gcGFyc2Uobm9kZS5vcGVyYW5kKS5pdGVtcygpfQogICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZSwgYXN0LkJpbk9wKToKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShub2RlLm9wLCAoYXN0LkFkZCwgYXN0LlN1YikpOgogICAgICAgICAgICAgICAgcmV0dXJuIGFkZChwYXJzZShub2RlLmxlZnQpLCBwYXJzZShub2RlLnJpZ2h0KSwgLTEgaWYgaXNpbnN0YW5jZShub2RlLm9wLCBhc3QuU3ViKSBlbHNlIDEpCiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2Uobm9kZS5vcCwgYXN0Lk11bHQpOgogICAgICAgICAgICAgICAgcmV0dXJuIG11bChwYXJzZShub2RlLmxlZnQpLCBwYXJzZShub2RlLnJpZ2h0KSkKICAgICAgICAgICAgaWYgKGlzaW5zdGFuY2Uobm9kZS5vcCwgYXN0LlBvdykgYW5kIGlzaW5zdGFuY2Uobm9kZS5yaWdodCwgYXN0LkNvbnN0YW50KQogICAgICAgICAgICAgICAgICAgIGFuZCB0eXBlKG5vZGUucmlnaHQudmFsdWUpIGlzIGludCBhbmQgMCA8PSBub2RlLnJpZ2h0LnZhbHVlIDw9IDQpOgogICAgICAgICAgICAgICAgcmVzdWx0ID0ge1pFUk86IEYoMSl9CiAgICAgICAgICAgICAgICBiYXNlID0gcGFyc2Uobm9kZS5sZWZ0KQogICAgICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobm9kZS5yaWdodC52YWx1ZSk6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0ID0gbXVsKHJlc3VsdCwgYmFzZSkKICAgICAgICAgICAgICAgIHJldHVybiByZXN1bHQKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbmx5IGJvdW5kZWQgcmF0aW9uYWwgcG9seW5vbWlhbHMgYXJlIHN1cHBvcnRlZDsgbm8gY29kZSBleGVjdXRpb24iKQoKICAgIHJldHVybiBwYXJzZSh0cmVlLmJvZHkpCgoKVEFSR0VUUyA9IHsKICAgICJzcXVhcmVfZXhhY3QiOiBwb2x5KCIoeCt1KSoqMi14KioyIiksCiAgICAicHJvZHVjdF9leGFjdCI6IHBvbHkoIih4K3UpKih5K3YpLXgqeSIpLAogICAgImNvbXBvc2l0aW9uIjogcG9seSgiKCh4K3UpKih5K3YpKSoqMi0oeCp5KSoqMiIpLAp9CldFSUdIVFMgPSB7ImxvY2FsIjogRigzLCAxMCksICJzcXVhcmVfZXhhY3QiOiBGKDEsIDUpLAogICAgICAgICAgICJwcm9kdWN0X2V4YWN0IjogRigxLCA1KSwgImNvbXBvc2l0aW9uIjogRigzLCAxMCl9CgoKQGRhdGFjbGFzcwpjbGFzcyBMZWRnZXI6CiAgICBjcmVkaXQ6IGRpY3QgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIGV2aWRlbmNlOiBkaWN0ID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgogICAgZGVmIHNjb3JlKHNlbGYpOgogICAgICAgIHJldHVybiBzdW0oc2VsZi5jcmVkaXQudmFsdWVzKCksIEYoMCkpCgogICAgZGVmIGF0dGVtcHQoc2VsZiwga2luZCwgZXhwcmVzc2lvbiwgKiwgcmFkaXVzPU5vbmUsIHRvbGVyYW5jZT1Ob25lLAogICAgICAgICAgICAgICAgc3RhdHVzPSJhc3NlcnRlZCIsIHRlYWNoZXJfYWdyZWVzPUZhbHNlLCBjb3N0PUYoMSwgNTApKToKICAgICAgICBiZWZvcmUgPSBzZWxmLnNjb3JlKCkKICAgICAgICBwcmlvcl9jcmVkaXQgPSBzZWxmLmNyZWRpdC5nZXQoa2luZCwgRigwKSkKICAgICAgICBhY2NlcHRlZCwgcmVhc29uID0gRmFsc2UsICIiCiAgICAgICAgaWYgc3RhdHVzID09ICJjb25qZWN0dXJlIjoKICAgICAgICAgICAgcmVhc29uID0gIkNvbmplY3R1cmUgcmV0YWluZWQgb3V0c2lkZSB2ZXJpZmllZCBsZWRnZXI7IG5vIHRydXRoIGNyZWRpdCIKICAgICAgICBlbGlmIHN0YXR1cyAhPSAiYXNzZXJ0ZWQiOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVbnN1cHBvcnRlZCBjbGFpbSBzdGF0dXMiKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNhbmRpZGF0ZSA9IHBvbHkoZXhwcmVzc2lvbikKICAgICAgICAgICAgaWYga2luZCA9PSAibG9jYWwiOgogICAgICAgICAgICAgICAgcmFkaXVzLCB0b2xlcmFuY2UgPSBGKHJhZGl1cyksIEYodG9sZXJhbmNlKQogICAgICAgICAgICAgICAgcmVzaWR1YWwgPSBhZGQoVEFSR0VUU1sic3F1YXJlX2V4YWN0Il0sIGNhbmRpZGF0ZSwgLTEpCiAgICAgICAgICAgICAgICBhY2NlcHRlZCA9IChyZXNpZHVhbCA9PSBwb2x5KCJ1KioyIikgYW5kIHJhZGl1cyA+IDAKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGFuZCB0b2xlcmFuY2UgPT0gRigxLCAxMDApIGFuZCByYWRpdXMqKjIgPD0gdG9sZXJhbmNlKQogICAgICAgICAgICAgICAgaWYgYWNjZXB0ZWQ6CiAgICAgICAgICAgICAgICAgICAgIyBGaXhlZCBzY29wZSBiYW5kczsgdGlueSBzY29wZSBjYW5ub3QgY2xhaW0gZnVsbCBjb3ZlcmFnZS4KICAgICAgICAgICAgICAgICAgICBhbW91bnQgPSBGKDMsIDEwKSBpZiByYWRpdXMgPj0gRigxLCAxMCkgZWxzZSBGKDEsIDEwKSBpZiByYWRpdXMgPj0gRigxLCAxMDApIGVsc2UgRigwKQogICAgICAgICAgICAgICAgICAgIHNlbGYuY3JlZGl0W2tpbmRdID0gbWF4KHNlbGYuY3JlZGl0LmdldChraW5kLCBGKDApKSwgYW1vdW50KQogICAgICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJFeGFjdCByZXNpZHVhbCB1XjI7IHJhdGlvbmFsIGludGVydmFsIGJvdW5kIHByb3ZlcyBkZWNsYXJlZCBhYnNvbHV0ZSB0b2xlcmFuY2UiCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJSZXNpZHVhbCBjZXJ0aWZpY2F0ZSBvciBmaXhlZC10b2xlcmFuY2Ugc2NvcGUgaXMgaW52YWxpZCIKICAgICAgICAgICAgZWxpZiBraW5kIGluIFRBUkdFVFM6CiAgICAgICAgICAgICAgICBkZXBlbmRlbmNpZXNfcmVhZHkgPSBraW5kICE9ICJjb21wb3NpdGlvbiIgb3IgeyJzcXVhcmVfZXhhY3QiLCAicHJvZHVjdF9leGFjdCJ9Lmlzc3Vic2V0KHNlbGYuZXZpZGVuY2UpCiAgICAgICAgICAgICAgICBhY2NlcHRlZCA9IGNhbmRpZGF0ZSA9PSBUQVJHRVRTW2tpbmRdIGFuZCBkZXBlbmRlbmNpZXNfcmVhZHkKICAgICAgICAgICAgICAgIGlmIGFjY2VwdGVkOgogICAgICAgICAgICAgICAgICAgIHNlbGYuY3JlZGl0W2tpbmRdID0gV0VJR0hUU1traW5kXQogICAgICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJFeGFjdCBwb2x5bm9taWFsIGlkZW50aXR5OyBkZWNsYXJlZCBjb21wb3NpdGlvbiBkZXBlbmRlbmNpZXMgYXZhaWxhYmxlIgogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICByZWFzb24gPSAiSWRlbnRpdHkgZmFsc2Ugb3IgY29tcG9zaXRpb24gZGVwZW5kZW5jaWVzIG1pc3NpbmciCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVbmtub3duIGZpeGVkIG1vZGVsaW5nIG9ibGlnYXRpb24iKQogICAgICAgICAgICBpZiBhY2NlcHRlZCBhbmQgKGtpbmQgbm90IGluIHNlbGYuZXZpZGVuY2Ugb3Igc2VsZi5jcmVkaXQuZ2V0KGtpbmQsIEYoMCkpID4gcHJpb3JfY3JlZGl0KToKICAgICAgICAgICAgICAgIHNlbGYuZXZpZGVuY2Vba2luZF0gPSB7ImV4cHJlc3Npb24iOiBleHByZXNzaW9uLCAidmVyaWZpY2F0aW9uIjogcmVhc29ufQogICAgICAgICAgICAgICAgaWYga2luZCA9PSAibG9jYWwiOgogICAgICAgICAgICAgICAgICAgIHNlbGYuZXZpZGVuY2Vba2luZF0udXBkYXRlKHJhZGl1cz1zdHIocmFkaXVzKSwgYWJzb2x1dGVUb2xlcmFuY2U9c3RyKHRvbGVyYW5jZSkpCiAgICAgICAgcGVuYWx0eSA9IEYoMSwgMTApIGlmIHN0YXR1cyA9PSAiYXNzZXJ0ZWQiIGFuZCBub3QgYWNjZXB0ZWQgZWxzZSBGKDApCiAgICAgICAgYWZ0ZXIgPSBzZWxmLnNjb3JlKCkKICAgICAgICByZXdhcmQgPSBhZnRlciAtIGJlZm9yZSAtIGNvc3QgLSBwZW5hbHR5CiAgICAgICAgcmV0dXJuIGRpY3Qoa2luZD1raW5kLCBleHByZXNzaW9uPWV4cHJlc3Npb24sIHN0YXR1cz1zdGF0dXMsIHRlYWNoZXJBZ3JlZXM9dGVhY2hlcl9hZ3JlZXMsCiAgICAgICAgICAgICAgICAgICAgdGVhY2hlckFncmVlbWVudFVzZWRBc0V2aWRlbmNlPUZhbHNlLCBhY2NlcHRlZD1hY2NlcHRlZCwgcmVhc29uPXJlYXNvbiwKICAgICAgICAgICAgICAgICAgICBiZWZvcmU9ZmxvYXQoYmVmb3JlKSwgYWZ0ZXI9ZmxvYXQoYWZ0ZXIpLCB2ZXJpZmllZEdhaW49ZmxvYXQoYWZ0ZXIgLSBiZWZvcmUpLAogICAgICAgICAgICAgICAgICAgIGNvc3Q9ZmxvYXQoY29zdCksIHVuc3VwcG9ydGVkQXNzZXJ0aW9uUGVuYWx0eT1mbG9hdChwZW5hbHR5KSwgcmV3YXJkPWZsb2F0KHJld2FyZCkpCgoKQGRhdGFjbGFzcwpjbGFzcyBSb3V0ZToKICAgIGV4cGVjdGVkX3JhdGU6IEYgPSBGKDApCiAgICBtaXNzaW5nOiBzZXQgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9c2V0KQogICAgcHJvYmVfdmVyc2lvbnM6IHNldCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1zZXQpCiAgICBwYXVzZWQ6IGJvb2wgPSBGYWxzZQoKICAgIGRlZiBmZWVkYmFjayhzZWxmLCB2ZXJpZmllZF9nYWluLCBjb3N0KToKICAgICAgICBzZWxmLmV4cGVjdGVkX3JhdGUgPSAoc2VsZi5leHBlY3RlZF9yYXRlICsgdmVyaWZpZWRfZ2FpbiAvIGNvc3QpIC8gMgoKICAgIGRlZiBuZXdfdG9vbHMoc2VsZiwgdG9vbHMpOgogICAgICAgICMgdG9vbHMgbWFwcyB2ZXJpZmllZCBkZXBlbmRlbmNpZXMgdG8gdGhlaXIgY29udGVudC9ldmlkZW5jZSBmaW5nZXJwcmludHMuCiAgICAgICAgdmVyc2lvbiA9IHR1cGxlKHNvcnRlZCgobmFtZSwgdG9vbHMuZ2V0KG5hbWUpKSBmb3IgbmFtZSBpbiBzZWxmLm1pc3NpbmcpKQogICAgICAgIGlmIHNlbGYucGF1c2VkIGFuZCBzZWxmLm1pc3NpbmcuaXNzdWJzZXQodG9vbHMpIGFuZCB2ZXJzaW9uIG5vdCBpbiBzZWxmLnByb2JlX3ZlcnNpb25zOgogICAgICAgICAgICBzZWxmLnByb2JlX3ZlcnNpb25zLmFkZCh2ZXJzaW9uKQogICAgICAgICAgICByZXR1cm4gMSAgIyBPbmUgYm91bmRlZCB0cmlhbDsgbm8gcGVybWFuZW50IHZhbHVlIGluY3JlYXNlLgogICAgICAgIHJldHVybiAwCgoKZGVmIHJ1bigpOgogICAgbGVkZ2VyID0gTGVkZ2VyKCkKICAgIHN0ZXBzID0gW10KICAgIHN0ZXBzLmFwcGVuZChsZWRnZXIuYXR0ZW1wdCgic3F1YXJlX2V4YWN0IiwgIjIqeCp1IiwgdGVhY2hlcl9hZ3JlZXM9VHJ1ZSkpCiAgICBhc3NlcnQgbm90IHN0ZXBzWy0xXVsiYWNjZXB0ZWQiXSBhbmQgc3RlcHNbLTFdWyJyZXdhcmQiXSA8IDAKICAgIHN0ZXBzLmFwcGVuZChsZWRnZXIuYXR0ZW1wdCgibG9jYWwiLCAiMip4KnUiLCByYWRpdXM9IjEvMTAwIiwgdG9sZXJhbmNlPSIxLzEwMCIpKQogICAgYXNzZXJ0IHN0ZXBzWy0xXVsiYWNjZXB0ZWQiXSBhbmQgc3RlcHNbLTFdWyJhZnRlciJdID09IC4xCiAgICBzdGVwcy5hcHBlbmQobGVkZ2VyLmF0dGVtcHQoImxvY2FsIiwgInUqeCt4KnUiLCByYWRpdXM9IjEvMTAwIiwgdG9sZXJhbmNlPSIxLzEwMCIpKQogICAgYXNzZXJ0IHN0ZXBzWy0xXVsiYWNjZXB0ZWQiXSBhbmQgc3RlcHNbLTFdWyJ2ZXJpZmllZEdhaW4iXSA9PSAwCiAgICBzdGVwcy5hcHBlbmQobGVkZ2VyLmF0dGVtcHQoImxvY2FsIiwgIjIqeCp1IiwgcmFkaXVzPSIxLzEwIiwgdG9sZXJhbmNlPSIxLzEwMCIpKQogICAgYXNzZXJ0IHN0ZXBzWy0xXVsiYWNjZXB0ZWQiXSBhbmQgc3RlcHNbLTFdWyJ2ZXJpZmllZEdhaW4iXSA9PSAuMgogICAgc3RlcHMuYXBwZW5kKGxlZGdlci5hdHRlbXB0KCJsb2NhbCIsICIyKngqdSIsIHJhZGl1cz0iMS81IiwgdG9sZXJhbmNlPSIxLzEwMCIpKQogICAgYXNzZXJ0IG5vdCBzdGVwc1stMV1bImFjY2VwdGVkIl0gYW5kIHN0ZXBzWy0xXVsidmVyaWZpZWRHYWluIl0gPT0gMAogICAgc3RlcHMuYXBwZW5kKGxlZGdlci5hdHRlbXB0KCJzcXVhcmVfZXhhY3QiLCAiMip4KnUrdSoqMiIpKQogICAgc3RlcHMuYXBwZW5kKGxlZGdlci5hdHRlbXB0KCJwcm9kdWN0X2V4YWN0IiwgIngqdit5KnUrdSp2IikpCiAgICBjb21wb3NpdGUgPSAiMip4KnkqKHgqdit5KnUrdSp2KSsoeCp2K3kqdSt1KnYpKioyIgogICAgc3RlcHMuYXBwZW5kKGxlZGdlci5hdHRlbXB0KCJjb21wb3NpdGlvbiIsIGNvbXBvc2l0ZSkpCiAgICBhc3NlcnQgc3RlcHNbLTFdWyJhY2NlcHRlZCJdIGFuZCBzdGVwc1stMV1bInZlcmlmaWVkR2FpbiJdID09IC4zCiAgICBhc3NlcnQgbGVkZ2VyLnNjb3JlKCkgPT0gMQogICAgYmxvY2tlZCA9IExlZGdlcigpLmF0dGVtcHQoImNvbXBvc2l0aW9uIiwgY29tcG9zaXRlKQogICAgYXNzZXJ0IG5vdCBibG9ja2VkWyJhY2NlcHRlZCJdCiAgICBob25lc3QgPSBMZWRnZXIoKS5hdHRlbXB0KCJzcXVhcmVfZXhhY3QiLCAiMip4KnUiLCBzdGF0dXM9ImNvbmplY3R1cmUiKQogICAgYXNzZXJ0IGhvbmVzdFsidW5zdXBwb3J0ZWRBc3NlcnRpb25QZW5hbHR5Il0gPT0gMCBhbmQgaG9uZXN0WyJ2ZXJpZmllZEdhaW4iXSA9PSAwCiAgICBkaXJlY3QgPSBMZWRnZXIoKS5hdHRlbXB0KCJzcXVhcmVfZXhhY3QiLCAiMip4KnUrdSoqMiIpCiAgICBjb3JyZWN0aW9uID0gTGVkZ2VyKCkKICAgIHdyb25nID0gY29ycmVjdGlvbi5hdHRlbXB0KCJzcXVhcmVfZXhhY3QiLCAiMip4KnUiKQogICAgZml4ZWQgPSBjb3JyZWN0aW9uLmF0dGVtcHQoInNxdWFyZV9leGFjdCIsICIyKngqdSt1KioyIikKICAgIGFzc2VydCB3cm9uZ1sicmV3YXJkIl0gKyBmaXhlZFsicmV3YXJkIl0gPCBkaXJlY3RbInJld2FyZCJdCiAgICAjIFUgZGlmZmVyZW5jZXMgdGVsZXNjb3BlOyBkZWxldGluZyB0aGVuIHJlc3RvcmluZyBjYW5ub3QgZmFybSBuZXQgcmV3YXJkLgogICAgZmFybSA9IExlZGdlcigpCiAgICBmYXJtLmF0dGVtcHQoInNxdWFyZV9leGFjdCIsICIyKngqdSt1KioyIikKICAgIGluaXRpYWwgPSBmYXJtLnNjb3JlKCkKICAgIGZhcm0uY3JlZGl0LnBvcCgic3F1YXJlX2V4YWN0IikKICAgIGRlbGV0aW9uX3Jld2FyZCA9IGZhcm0uc2NvcmUoKSAtIGluaXRpYWwgLSBGKDEsIDUwKQogICAgcmVzdG9yZWQgPSBmYXJtLmF0dGVtcHQoInNxdWFyZV9leGFjdCIsICIyKngqdSt1KioyIikKICAgIGFzc2VydCBmbG9hdChkZWxldGlvbl9yZXdhcmQpICsgcmVzdG9yZWRbInJld2FyZCJdIDwgMAogICAgIyBFeGFjdCBpZGVudGl0aWVzIGNoZWNrZWQgZm9yIGFsbCByZWFsczsgdGhlc2UgdmFsdWVzIGFyZSBpbmRlcGVuZGVudCBzbW9rZSBleGFtcGxlcy4KICAgIGRlZiBldmFsdWF0ZShwLCB2YWx1ZXMpOgogICAgICAgIHJldHVybiBzdW0oYyAqIHZhbHVlc1swXSoqa1swXSAqIHZhbHVlc1sxXSoqa1sxXSAqIHZhbHVlc1syXSoqa1syXSAqIHZhbHVlc1szXSoqa1szXSBmb3IgaywgYyBpbiBwLml0ZW1zKCkpCiAgICBhc3NlcnQgZXZhbHVhdGUoVEFSR0VUU1sic3F1YXJlX2V4YWN0Il0sICgxLCAwLCAxLCAwKSkgPT0gMwogICAgYXNzZXJ0IGV2YWx1YXRlKHBvbHkoIjIqeCp1IiksICgxLCAwLCAxLCAwKSkgPT0gMgogICAgYXNzZXJ0IGV2YWx1YXRlKFRBUkdFVFNbImNvbXBvc2l0aW9uIl0sICgyLCAzLCAxLCAtMSkpID09IDAKICAgIHJvdXRlID0gUm91dGUobWlzc2luZz17InByb2R1Y3RfZXhhY3QifSkKICAgIHJvdXRlLmZlZWRiYWNrKEYoMSwgMTApLCBGKDEsIDUwKSkKICAgIHJhdGVzID0gW2Zsb2F0KHJvdXRlLmV4cGVjdGVkX3JhdGUpXQogICAgZm9yIF8gaW4gcmFuZ2UoNCk6CiAgICAgICAgcm91dGUuZmVlZGJhY2soRigwKSwgRigxLCA1MCkpCiAgICAgICAgcmF0ZXMuYXBwZW5kKGZsb2F0KHJvdXRlLmV4cGVjdGVkX3JhdGUpKQogICAgYXNzZXJ0IHJhdGVzWy0xXSA8IHJhdGVzWzBdCiAgICBzYXZlZF9ldmlkZW5jZSA9IGpzb24uZHVtcHMobGVkZ2VyLmV2aWRlbmNlLCBzb3J0X2tleXM9VHJ1ZSkKICAgIHJvdXRlLnBhdXNlZCA9IFRydWUKICAgIGFzc2VydCBzYXZlZF9ldmlkZW5jZSA9PSBqc29uLmR1bXBzKGxlZGdlci5ldmlkZW5jZSwgc29ydF9rZXlzPVRydWUpCiAgICBhc3NlcnQgcm91dGUubmV3X3Rvb2xzKHsic3F1YXJlX2V4YWN0IjogInNxdWFyZS12MSJ9KSA9PSAwCiAgICBhc3NlcnQgcm91dGUubmV3X3Rvb2xzKHsic3F1YXJlX2V4YWN0IjogInNxdWFyZS12MSIsICJwcm9kdWN0X2V4YWN0IjogInByb2R1Y3QtdjEifSkgPT0gMQogICAgYXNzZXJ0IHJvdXRlLm5ld190b29scyh7InNxdWFyZV9leGFjdCI6ICJzcXVhcmUtdjEiLCAicHJvZHVjdF9leGFjdCI6ICJwcm9kdWN0LXYxIn0pID09IDAKICAgIGFzc2VydCByb3V0ZS5uZXdfdG9vbHMoeyJ1bnJlbGF0ZWQiOiAidW5yZWxhdGVkLXYyIiwgInByb2R1Y3RfZXhhY3QiOiAicHJvZHVjdC12MSJ9KSA9PSAwCiAgICBhc3NlcnQgcm91dGUubmV3X3Rvb2xzKHsicHJvZHVjdF9leGFjdCI6ICJwcm9kdWN0LXYyIn0pID09IDEKICAgICMgQSBsYXRlciBuYXJyb3cgdmFsaWQgcmVzdGF0ZW1lbnQgbXVzdCBub3QgZGlzY2FyZCB0aGUgd2lkZXIgcmFuZ2UncyBjZXJ0aWZpY2F0ZS4KICAgIHJldGFpbmVkX2xvY2FsID0gZGljdChsZWRnZXIuZXZpZGVuY2VbImxvY2FsIl0pCiAgICBuYXJyb3cgPSBsZWRnZXIuYXR0ZW1wdCgibG9jYWwiLCAiMip4KnUiLCByYWRpdXM9IjEvMTAwIiwgdG9sZXJhbmNlPSIxLzEwMCIpCiAgICBhc3NlcnQgbmFycm93WyJ2ZXJpZmllZEdhaW4iXSA9PSAwIGFuZCBsZWRnZXIuZXZpZGVuY2VbImxvY2FsIl0gPT0gcmV0YWluZWRfbG9jYWwKICAgICMgUGFyc2VyIGRvZXMgbm90IGV4ZWN1dGUgY2FsbHMgb3IgYWNjZXB0IGFyYml0cmFyeSBuYW1lcy4KICAgIGZvciB1bnNhZmUgaW4gWyJmKHgpIiwgInguX19jbGFzc19fIiwgInoreCIsICJ4Kio5OSJdOgogICAgICAgIHRyeToKICAgICAgICAgICAgcG9seSh1bnNhZmUpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigiVW5zdXBwb3J0ZWQgZXhwcmVzc2lvbiB3YXMgYWNjZXB0ZWQiKQogICAgcmV0dXJuIGRpY3Qoc3RhdHVzPSJkcnlfcnVuX2NoZWNrc19wYXNzZWQiLCBydW5uZXJTaGEyNTY9aGFzaGxpYi5zaGEyNTYoUGF0aChfX2ZpbGVfXykucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAgICAgICAgIG5ldXJhbFRyYWluaW5nVXBkYXRlcz0wLCB0ZWFjaGVyQ2FsbHM9MCwKICAgICAgICAgICAgICAgIHN0dWRlbnRHZW5lcmF0ZWRDbGFpbXM9RmFsc2UsIG5vdmVsdHlDbGFpbT1GYWxzZSwKICAgICAgICAgICAgICAgIHNjb3BlPSJoYW5kLXdyaXR0ZW4gZml4dHVyZXM7IGV4YWN0IGZpbml0ZS1kb21haW4gcmV3YXJkIGFuZCBzY2hlZHVsaW5nIGNoZWNrcywgbm90IGVmZmljYWN5IGV2aWRlbmNlIiwKICAgICAgICAgICAgICAgIHNjb3JlV2VpZ2h0cz17azogZmxvYXQodikgZm9yIGssIHYgaW4gV0VJR0hUUy5pdGVtcygpfSwgc3RlcHM9c3RlcHMsCiAgICAgICAgICAgICAgICBmaW5hbFZlcmlmaWVkQ292ZXJhZ2U9ZmxvYXQobGVkZ2VyLnNjb3JlKCkpLCBzdGFsbGVkUm91dGVSYXRlcz1yYXRlcywKICAgICAgICAgICAgICAgIGNoZWNrcz1bInRlYWNoZXIgYWdyZWVtZW50IGNhbm5vdCBjZXJ0aWZ5IGEgZmFsc2UgaWRlbnRpdHkiLCAidmFsaWQgbG9jYWwgbW9kZWwgcmVjZWl2ZXMgcGFydGlhbCBjcmVkaXQiLAogICAgICAgICAgICAgICAgICAgICAgICAiZXF1aXZhbGVudCByZXdvcmRpbmcgcmVjZWl2ZXMgemVybyBuZXcgdHJ1dGggY3JlZGl0IiwgIndpZGVyIHZhbGlkIHNjb3BlIGluY3JlYXNlcyBjb3ZlcmFnZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICJpbnZhbGlkIHNjb3BlIGVubGFyZ2VtZW50IHJlamVjdGVkIiwgInZlcmlmaWVkIGNvbXBvc2l0aW9uIGFkZHMgYSBuZXcgb2JsaWdhdGlvbiIsCiAgICAgICAgICAgICAgICAgICAgICAgICJjb21wb3NpdGlvbiB3aXRob3V0IGRlcGVuZGVuY2llcyByZWplY3RlZCIsICJob25lc3QgY29uamVjdHVyZSBoYXMgbm8gZmFsc2UtYXNzZXJ0aW9uIHBlbmFsdHkiLAogICAgICAgICAgICAgICAgICAgICAgICAiaW50ZW50aW9uYWwgZXJyb3IgdGhlbiByZXBhaXIgc2NvcmVzIGJlbG93IGNvcnJlY3QgZmlyc3QgYXR0ZW1wdCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJkZWxldGUvcmUtYWRkIGhhcyBubyBwb3NpdGl2ZSBuZXQgcmV3YXJkIiwgInN0YWxsaW5nIHJlZHVjZXMgcHJpb3JpdHkgd2l0aG91dCBkZWxldGluZyBrbm93bGVkZ2UiLAogICAgICAgICAgICAgICAgICAgICAgICAibmV3IHJlbGV2YW50IGRlcGVuZGVuY3kgcGVybWl0cyBvbmUgYm91bmRlZCByZXZpc2l0IHBlciBldmlkZW5jZSBmaW5nZXJwcmludCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJuYXJyb3cgcmVzdGF0ZW1lbnQgY2Fubm90IGRpc2NhcmQgYnJvYWRlciBjb3ZlcmFnZSBjZXJ0aWZpY2F0ZSIsICJ1bnN1cHBvcnRlZCBzeW50YXggcmVqZWN0ZWQiXSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCB0eXBlPVBhdGgpCiAgICBhcmdzID0gcC5wYXJzZV9hcmdzKCkKICAgIHJlc3VsdCA9IHJ1bigpCiAgICBpZiBhcmdzLm91dDoKICAgICAgICB3aXRoIGFyZ3Mub3V0Lm9wZW4oIngiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBzdHJlYW06CiAgICAgICAgICAgIGpzb24uZHVtcChyZXN1bHQsIHN0cmVhbSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiwgYWxsb3dfbmFuPUZhbHNlKQogICAgICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgIHByaW50KGpzb24uZHVtcHMoe2s6IHJlc3VsdFtrXSBmb3IgayBpbiBbInN0YXR1cyIsICJuZXVyYWxUcmFpbmluZ1VwZGF0ZXMiLCAidGVhY2hlckNhbGxzIiwgInN0dWRlbnRHZW5lcmF0ZWRDbGFpbXMiXX0pKQogICAgcHJpbnQoZiJDaGVja3MgcGFzc2VkOiB7bGVuKHJlc3VsdFsnY2hlY2tzJ10pfS4gTm8gc3R1ZGVudC1sZWFybmluZyByZXN1bHQgaXMgY2xhaW1lZC4iKQo="}}''')
for name, record in PAYLOAD.items():
    data = base64.b64decode(record['base64'])
    assert hashlib.sha256(data).hexdigest() == record['sha256']
    target = TOOLS / name
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).hexdigest() == record['sha256'], '已有脚本版本不同，请换 WORK 目录'
    else:
        target.write_bytes(data)
print('脚本就绪：', WORK)

def run(arguments, timeout=900):
    try:
        subprocess.run([sys.executable, '-u', str(TOOLS/'zip_pilot.py')] + [str(x) for x in arguments],
                       check=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError('达到进程时间上限。已完成记录保留；没有自动重启。')

PLAN = WORK / ('plan-' + uuid.uuid4().hex[:8])
run(['prepare', '--out', PLAN], timeout=30)


## 2. 填好配置（先不要申请 GPU）

学生选用 `swiss-ai/Apertus-v1.1-0.5B-Instruct`。
固定版本 `a140fd61fb57422c36a26301caea17edee799874`，使用完整 BF16 safetensors，
不使用推理专用量化文件。原 17M GPT 不用于此实验。
`STUDENT_PATH` 指向完整快照；已有文件可直接填写，或开启下面的独立下载步骤。
老师用独立 Apertus 聊天服务，活动 CSCS 推理或另行部署均可。
`TEACHER_BASE_URL` 是 /v1 这类 API 前缀；实际服务模型名按服务填写。
老师部署身份先声明并保存，客户端不能独立验证权重版本。
这些配置可以保存在 Notebook；密钥不要写进 Notebook。


In [ ]:
STUDENT_ID = 'swiss-ai/Apertus-v1.1-0.5B-Instruct'
STUDENT_REVISION = 'a140fd61fb57422c36a26301caea17edee799874'
STUDENT_PATH = WORK / 'models' / 'Apertus-v1.1-0.5B-Instruct'
RUN_DOWNLOAD = False  # 只下载文件；不申请 GPU、不推理、不调用老师
DOWNLOAD_ENDPOINT = 'https://huggingface.co'  # 可手动选 https://hf-mirror.com；下载后仍验证官方哈希
TEACHER_BASE_URL = ''
TEACHER_MODEL = 'swiss-ai/Apertus-v1.5-8B'
TEACHER_REVISION = ''  # 服务部署版本标签；不冒充自动核实
DEVICE = 'cuda'       # CPU 准备时也可改为 'cpu'
COLLECT_DEVICE = 'cuda'  # 老师占同一 GPU 时改 'cpu'，或用本地学生+远程老师
MODE = 'zip'          # 对照可选 'none' 或 'text'
SEED = 1337          # 后续多种子：1337, 2027, 4099
COLLECTION = WORK / ('collection-' + uuid.uuid4().hex[:8])
TRAIN_RUN = WORK / ('train-' + MODE + '-' + uuid.uuid4().hex[:8])
TEACHER_CACHE = WORK / 'teacher-cache'
RUN_COLLECT = False
RUN_BASELINE = False  # 先做学生起点检查；没有老师调用
BASELINE_RUN = WORK / ('baseline-' + uuid.uuid4().hex[:8])
RUN_TRAIN = False
RUN_FINAL = False     # 设置锁定前不要打开最终公开开发测试


## 2.5. 可选：下载固定版本的学生（CPU 环境即可）

默认关闭。没有快照时，先安装下一节的依赖，再把 `RUN_DOWNLOAD` 改为 True，
单独运行此格。它只请求官方模型文件，匿名下载，不调用推理或老师；需要网络和磁盘。
不自动安装 PyTorch，不自动重试整个任务。中断后的缓存可用于下次手动重跑。
只有七个必需文件的大小和官方固定提交的内容哈希均通过，才显示完成。
目录已存在或下载函数正常返回不代表快照完整；断网返回不完整缓存会报错。
官方入口不可访问时可手动设置 `DOWNLOAD_ENDPOINT='https://hf-mirror.com'`。
这是第三方文件镜像，不是官方服务；匿名请求，仍按官方提交的全部文件哈希核验。


In [ ]:
if RUN_DOWNLOAD:
    import textwrap
    download_code = textwrap.dedent("""
        import hashlib, json, sys
        from pathlib import Path
        from huggingface_hub import snapshot_download
        EXPECTED_FILES = {
            'config.json': {'size': 853, 'gitSha1': 'fed97bcf53909198ea671fa6c39071d58150b9ae'},
            'generation_config.json': {'size': 144, 'gitSha1': 'f042bd90602218eab507817019126824a5a0d211'},
            'model.safetensors': {'size': 1145165288, 'sha256': 'b7fda2f3e6a47bc00de20f88f38ab52fb2d1df68bf9b4c0016a99dd4d2aa8b3d'},
            'tokenizer.json': {'size': 17078368, 'sha256': 'be12f4375d655cc740864e3a9041bcddd8477942f209d9e7f27f6c8767162638'},
            'tokenizer_config.json': {'size': 177274, 'gitSha1': 'b4c05699fb3617941f2c5a0030b8d6a2dfcf9424'},
            'special_tokens_map.json': {'size': 560, 'gitSha1': 'aada9e1ceb41556521fda130b61871ad61e17015'},
            'chat_template.jinja': {'size': 5250, 'gitSha1': 'a46508148deb7081bfcd25e89735a51226b2426b'},
        }

        def verify_files(directory, expected):
            missing = [name for name in expected if not (directory/name).is_file()]
            if missing:
                raise ValueError('学生快照不完整；缺少文件：' + ', '.join(missing))
            for name, record in expected.items():
                file = directory/name
                size = file.stat().st_size
                if size != record['size']:
                    raise ValueError('学生文件大小不匹配：' + name)
                if 'sha256' in record:
                    digest = hashlib.sha256()
                    wanted = record['sha256']
                else:
                    digest = hashlib.sha1()
                    digest.update(('blob ' + str(size)).encode() + bytes([0]))
                    wanted = record['gitSha1']
                with file.open('rb') as stream:
                    for block in iter(lambda: stream.read(1024 * 1024), b''):
                        digest.update(block)
                if digest.hexdigest() != wanted:
                    raise ValueError('学生文件内容哈希不匹配：' + name)

        model_id, revision, output, endpoint = sys.argv[1:]
        if endpoint not in {'https://huggingface.co', 'https://hf-mirror.com'}:
            raise ValueError('下载入口只支持官方 Hub 或显式选择的文件镜像')
        if (model_id, revision) != ('swiss-ai/Apertus-v1.1-0.5B-Instruct',
                                   'a140fd61fb57422c36a26301caea17edee799874'):
            raise ValueError('此下载格只验证固定版本的 Apertus Mini；请勿替换模型或版本')
        destination = Path(output)
        identity = dict(modelId=model_id, revision=revision)
        marker = destination/'lain-snapshot.json'
        if destination.exists() and any(destination.iterdir()):
            if not marker.is_file() or json.loads(marker.read_text()) != identity:
                raise ValueError('下载目标包含其它版本；请使用新的 STUDENT_PATH')
        destination.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps(identity))
        snapshot_download(repo_id=model_id, revision=revision, token=False, local_dir=destination, endpoint=endpoint,
            allow_patterns=list(EXPECTED_FILES), max_workers=2)
        verify_files(destination, EXPECTED_FILES)
        print('Verified all 7 files for fixed revision', revision)
    """)
    endpoint = globals().get('DOWNLOAD_ENDPOINT', 'https://huggingface.co')
    download_env = os.environ.copy()
    download_env['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
    if endpoint == 'https://hf-mirror.com':
        download_env['HF_HUB_DISABLE_XET'] = '1'
    subprocess.run([globals().get('PY', sys.executable), '-u', '-c', download_code,
                    STUDENT_ID, STUDENT_REVISION, str(STUDENT_PATH), endpoint],
                   check=True, timeout=900, env=download_env)
    print('学生快照完整且校验通过：', STUDENT_PATH)
else:
    print('没有下载；可使用已有完整快照。')


## 3. 检查环境（不加载权重）

需要已有 PyTorch；学生使用 Transformers 4.57.6。缺依赖时先在独立环境
安装 `python -m pip install transformers==4.57.6`，不替换已有 CUDA PyTorch。
老师使用官方 Apertus 1.5 独立环境/容器，不混装到学生环境。
输出 GPU 实际型号、显存和 BF16 支持后再决定部署；不假设魔搭分配了什么卡。


In [ ]:
run(['preflight', '--student-path', STUDENT_PATH], timeout=60)


## 3.5. 学生起点（4 次短推理，老师调用为零）

只在学生快照和算力已经就绪时开启 `RUN_BASELINE`。查看中文/英文原始输出，
再看对象输出是否符合格式。格式通过不等于能力已经被自动认证；格式失败
会在调用老师前停止，先检查语言起点或提示。不要放宽公式核验来制造成功。


In [ ]:
if RUN_BASELINE:
    run(['baseline', '--student-path', STUDENT_PATH, '--device', DEVICE, '--out', BASELINE_RUN])
else:
    print('起点推理未启动。没有老师调用。')
BASELINE = BASELINE_RUN / 'baseline.json'


## 4. 少量收集：最多 6 次 Apertus 网络调用

确认学生能表达对象、快照存在、老师服务可用后，才把 `RUN_COLLECT` 改为 True。
若需要密钥，下方会用不可见输入放入环境变量，内容不会保存到 Notebook。
相同请求缓存复用；失败不重试。没有接纳的公式就停止，先检查起点和格式。
同一 GPU 部署老师时先完成材料收集，关闭老师后才训练学生。


In [ ]:
if RUN_COLLECT:
    assert STUDENT_PATH.is_dir(), '先准备学生完整快照'
    assert BASELINE.is_file(), '先运行起点检查并查看输出'
    assert TEACHER_BASE_URL and TEACHER_REVISION, '先填写老师接口和部署版本'
    if 'LAIN_TEACHER_API_KEY' not in os.environ:
        from getpass import getpass
        secret = getpass('Apertus key（本地无鉴权服务可留空）：')
        if secret:
            os.environ['LAIN_TEACHER_API_KEY'] = secret
        del secret
    run(['collect', '--student-path', STUDENT_PATH, '--device', COLLECT_DEVICE,
         '--teacher-base-url', TEACHER_BASE_URL, '--teacher-model', TEACHER_MODEL,
         '--teacher-revision', TEACHER_REVISION, '--teacher-cache', TEACHER_CACHE,
         '--baseline', BASELINE,
         '--out', COLLECTION, '--max-teacher-calls', 6, '--max-seconds', 600])
else:
    print('收集未启动。没有老师调用。')


## 5. 查看收集结果，再训练

不显示密钥，也不自动显示完整老师反馈。初稿和提示后修订要分开看；
只有公式和范围被程序核验，解释文本不等于模型内在思想或已证明的理解。


In [ ]:
CORPUS = COLLECTION / 'corpus.json'
if CORPUS.exists():
    corpus = json.loads(CORPUS.read_text())
    print('有效 zip：', len(corpus['notes']), '老师网络调用：', corpus['teacherNetworkCalls'],
          '缓存命中：', corpus['teacherCacheHits'])
    for record in corpus['records']:
        print(record['task']['key'], '独立初稿覆盖', record['initialCheck']['score'],
              '提示后覆盖', record['revisionCheck']['score'])
else:
    print('尚未收集。保持不训练。')


## 6. 学生短训练：4 次接口热身 + 2 条两步策略轨迹

把 `RUN_TRAIN` 改为 True 才执行。老师调用为零；基础学生冻结，只更新新增模块。
zip 在隐藏层参与计算；text 对照把同一材料放到文本上下文；none 对照保留普通路径。
默认只是通路测试，生成预算和总运算尚未严格匹配，不作为效果胜负结论。
每次输出新建目录，不覆盖已有结果。超时保留完成记录，不自动继续。


In [ ]:
if RUN_TRAIN:
    assert CORPUS.is_file(), '先收集并查看核验材料'
    run(['train', '--student-path', STUDENT_PATH, '--device', DEVICE, '--corpus', CORPUS,
         '--out', TRAIN_RUN, '--mode', MODE, '--seed', SEED,
         '--warmup-steps', 4, '--pg-episodes', 2, '--max-seconds', 600])
    result = json.loads((TRAIN_RUN/'result.json').read_text())
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print('训练未启动。参数更新 0。')


## 7. 最终冻结评测（独立执行）

只有设置已经锁定时才把 `RUN_FINAL` 改为 True。无老师，无在线更新。
首次查看后不能反复用这些题调参再称未见测试；当前题目公开，只是开发验证。
训练报告中的关闭/错配 zip 先帮助检查内容依赖，不能靠单个种子断定收益。


In [ ]:
if RUN_FINAL:
    FINAL_RUN = WORK / ('final-' + uuid.uuid4().hex[:8])
    run(['final-eval', '--student-path', STUDENT_PATH, '--device', DEVICE,
         '--run', TRAIN_RUN, '--out', FINAL_RUN])
    print('最终记录：', FINAL_RUN/'final.json')
else:
    print('最终题未打开。')


## 保存结果后关闭云端 GPU

保留 `lain-zip-pilot` 目录中的 JSON、适配器 safetensors、archive 和老师缓存，
并记录 GPU 型号、实际耗时、额度及网络错误。Notebook 不替你关闭云端实例，
需要在魔搭控制台停止实例。原学生快照不被修改。

下一阶段：冻结材料并匹配总预算，多种子分别检验 SFT、进展奖励、文本/zip
以及调度的贡献。当前没有 Apertus 真实调用或 GPU 成功记录，也没有自我升级结论。
